# Federated Learning for Liver Cirrhosis Outcome Prediction (with XAI)
This notebook follows the 22 federated learning steps one by one.
Every step is marked with a **STEP number** so you can copy the matching text, table or figure into your paper.

| Part | Steps |
|---|---|
| Data and split | 3 Raw data, 4 Holdout split |
| Federated setup | 1 Server, 2 Clients, 5 Partitioning, 6 IID / Non-IID, 7 Dirichlet alpha, 8 Local preprocessing |
| Federated training | 9 Local training, 10 Local epochs, 11 Updates, 12 Quantization, 13 Differential privacy, 14 Secure upload, 15 Rounds, 16 Aggregation, 17 Global update, 18 Broadcast, 19 Convergence |
| Results | 20 Global evaluation, 21 Performance metrics |
| Explainability | 22 XAI (SHAP, LIME, permutation importance) |

## Cell 0: Install libraries
Run the cells from top to bottom (Runtime > Run all). When asked, upload cirrhosis.csv.

In [ ]:
!pip install -q shap lime

## Cell 1: Imports and plot style

In [ ]:
import os, glob, shutil, warnings, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import shap
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, log_loss)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300,
                     "axes.titleweight": "bold", "axes.titlesize": 13,
                     "axes.labelsize": 11, "legend.frameon": True})

# Colors used everywhere so that all figures look consistent
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
print("Libraries loaded. SHAP version:", shap.__version__)

## Cell 2: Settings (change values here only)
All important choices are in this one cell, so you can describe them in the "Experimental setup" section of your paper.

In [ ]:
SEED = 42                  # main random seed
N_REPEATS = 5              # number of repeated runs with different seeds (for mean +- std)

# ----- Data settings -----
TARGET_MODE = "binary"     # "binary" = Survived vs Died | "multiclass" = Alive / Transplant / Died
USE_N_DAYS = False         # follow-up days. Keep False for a fair baseline prediction (see notes at the end)

# ----- Federated settings -----
N_CLIENTS = 5              # number of hospitals / clients
ALPHA_NON_IID = 0.5        # Dirichlet alpha for Non-IID data (small alpha = more Non-IID)
ROUNDS = 40                # communication rounds
LOCAL_EPOCHS = 5           # local epochs on every client in every round
BATCH_SIZE = 16
LEARNING_RATE = 0.05
HIDDEN_UNITS = 16          # size of the hidden layer of the neural network
L2 = 1e-3                  # weight decay
MU_PROX = 0.05             # FedProx proximal strength
SERVER_LR = 1.0            # step size of the global update

# ----- Privacy and communication settings -----
QUANT_BITS = 8             # quantization bits for uploaded updates
DP_EPSILON = 10.0          # differential privacy epsilon (per round)
DP_DELTA = 1e-5
DP_CLIP = 1.0              # clipping norm of every client update
EPSILON_LIST = [1, 2, 5, 10, 20, 50]   # for the privacy-utility curve

AGGREGATORS = ["FedAvg", "FedProx", "FedNova"]
print("Settings ready.")

## Cell 3: Find the data file and create the output folder
(This cell is different in the Colab file and the Kaggle file.)

In [ ]:
from google.colab import files

CSV_PATH = None
for p in glob.glob("/content/**/cirrhosis.csv", recursive=True) + glob.glob("/content/drive/**/cirrhosis.csv", recursive=True):
    CSV_PATH = p
    break
if CSV_PATH is None:
    print("Please choose the file cirrhosis.csv from your computer")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
OUT_DIR = "/content/FL_Cirrhosis_Outputs"
print("Data file:", CSV_PATH)

## Cell 4: Helper functions to save every figure and table
Everything is saved in the output folder (PNG at 300 dpi for figures, CSV for tables), and zipped at the end.

In [ ]:
def save_fig(name):
    path = os.path.join(OUT_DIR, "figures", name + ".png")
    plt.savefig(path, bbox_inches="tight")
    plt.show()


def show_table(df, title, name, cmap="YlGnBu", fmt="{:.4f}", index=True):
    # shows a colorful table and saves it as CSV
    print("\n" + title)
    df.to_csv(os.path.join(OUT_DIR, "tables", name + ".csv"), index=index)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    try:
        styler = df.style
        if len(num_cols) > 0:
            styler = styler.background_gradient(cmap=cmap, axis=None, subset=num_cols).format(fmt, subset=num_cols)
        display(styler)
    except Exception:
        display(df)


os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "tables"), exist_ok=True)
print("Output folder:", OUT_DIR)

## STEP 3: Raw data
The raw data is the Mayo Clinic PBC liver cirrhosis table. In a real federated setting this table would be split across hospitals and could never be pooled.

In [ ]:
raw = pd.read_csv(CSV_PATH)
print("Raw data shape:", raw.shape)
display(raw.head(8))

summary = pd.DataFrame({
    "Type": raw.dtypes.astype(str),
    "Missing": raw.isna().sum(),
    "Missing %": (raw.isna().mean() * 100).round(1),
    "Unique values": raw.nunique(),
})
show_table(summary, "Table 1. Raw dataset description", "table01_dataset_description", cmap="PuBuGn", fmt="{:.1f}")

status_names = {"C": "Censored (alive)", "CL": "Censored (liver transplant)", "D": "Died"}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
counts = raw["Status"].map(status_names).value_counts()
ax[0].bar(counts.index, counts.values, color=PALETTE[:3], edgecolor="black")
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 3, str(v), ha="center", fontweight="bold")
ax[0].set_title("Outcome (Status) distribution"); ax[0].set_ylabel("Patients")
ax[0].tick_params(axis="x", rotation=15)

miss = raw.isna().mean().sort_values(ascending=False) * 100
miss = miss[miss > 0]
ax[1].barh(miss.index[::-1], miss.values[::-1], color=sns.color_palette("flare", len(miss)))
ax[1].set_title("Missing values (%)"); ax[1].set_xlabel("Percent missing")

for st, col in zip(["C", "CL", "D"], PALETTE[:3]):
    sns.kdeplot(raw.loc[raw["Status"] == st, "Age"] / 365.25, ax=ax[2], fill=True, color=col,
                label=status_names[st], alpha=0.35)
ax[2].set_title("Age (years) by outcome"); ax[2].set_xlabel("Age"); ax[2].legend()
plt.tight_layout()
save_fig("fig01_raw_data_overview")

## Feature preparation (simple cleaning that does not use any other patient's information)
Text values are changed to numbers, age is changed from days to years, and very skewed blood tests get a log transform.

In [ ]:
LOG_COLS = ["Bilirubin", "Cholesterol", "Alk_Phos", "SGOT", "Tryglicerides", "Copper", "Prothrombin", "Platelets"]


def make_features(raw_df):
    d = raw_df.copy()
    d = d.drop(columns=["ID"])
    d["Age"] = d["Age"] / 365.25
    d["Drug_missing"] = d["Drug"].isna().astype(int)        # patient was not in the drug trial
    d["Drug"] = d["Drug"].map({"Placebo": 0, "D-penicillamine": 1})
    d["Sex"] = d["Sex"].map({"F": 0, "M": 1})
    for c in ["Ascites", "Hepatomegaly", "Spiders"]:
        d[c] = d[c].map({"N": 0, "Y": 1})
    d["Edema"] = d["Edema"].map({"N": 0, "S": 1, "Y": 2})
    if not USE_N_DAYS:
        d = d.drop(columns=["N_Days"])
    status = d.pop("Status")
    for c in LOG_COLS:
        d[c] = np.log1p(d[c])
    return d, status


X_all, status_all = make_features(raw)

if TARGET_MODE == "binary":
    CLASS_NAMES = ["Survived", "Died"]
    y_all = (status_all == "D").astype(int)
else:
    CLASS_NAMES = ["Alive", "Transplant", "Died"]
    y_all = status_all.map({"C": 0, "CL": 1, "D": 2})

# Stage is known at the start (biopsy). Rows with unknown stage stay and are imputed later by each client.
FEATURES = X_all.columns.tolist()
N_CLASSES = len(CLASS_NAMES)
POS_CLASS = N_CLASSES - 1          # "Died" class, used for risk explanations
print("Features used (%d):" % len(FEATURES), FEATURES)
print("Classes:", CLASS_NAMES, "| counts:", y_all.value_counts().sort_index().tolist())

plt.figure(figsize=(10, 8))
corr = pd.concat([X_all, y_all.rename("Outcome")], axis=1).corr()
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=0.3, cbar_kws={"shrink": 0.7})
plt.title("Correlation heatmap of features and outcome")
save_fig("fig02_correlation_heatmap")

## STEP 4: Holdout split (Train / Validation / Test)
The split is done **once, before anything else**. Only the training part is shared out to clients.
* Validation set (15%) is kept by the server and is only used to pick the best communication round.
* Test set (15%) is kept by the server and is only used for the final global evaluation.

In [ ]:
def holdout_split(X, y, seed):
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
    X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=seed)
    return X_tr, X_va, X_te, y_tr, y_va, y_te


X_tr, X_va, X_te, y_tr, y_va, y_te = holdout_split(X_all, y_all, SEED)

split_table = pd.DataFrame({
    "Set": ["Train (to clients)", "Validation (server)", "Test (server)"],
    "Patients": [len(y_tr), len(y_va), len(y_te)],
})
for k, name in enumerate(CLASS_NAMES):
    split_table[name] = [int((y_tr == k).sum()), int((y_va == k).sum()), int((y_te == k).sum())]
split_table = split_table.set_index("Set")
show_table(split_table, "Table 2. Holdout split (70 / 15 / 15)", "table02_holdout_split", cmap="Greens", fmt="{:.0f}")

ax = split_table[CLASS_NAMES].plot(kind="bar", stacked=True, figsize=(8, 4.5), color=PALETTE[:N_CLASSES], edgecolor="black")
plt.title("Holdout split with class balance"); plt.ylabel("Patients"); plt.xticks(rotation=0)
for c in ax.containers:
    ax.bar_label(c, label_type="center", color="white", fontweight="bold")
save_fig("fig03_holdout_split")

## STEPS 5, 6, 7: Data partitioning, IID / Non-IID, Dirichlet distribution (alpha)
* **IID**: every client receives a random, equal, class-balanced share.
* **Non-IID**: for each class, a Dirichlet(alpha) draw decides how much of that class every client gets.
  Small alpha (0.1) gives very different clients, large alpha (10) is nearly IID.

In [ ]:
def partition_data(y, n_clients, alpha, seed, min_size=20):
    # alpha = None means IID. Returns a list of index arrays, one for every client.
    y = np.asarray(y)
    rng = np.random.RandomState(seed)
    classes = np.unique(y)
    best = None
    for attempt in range(500):
        idx_per_client = [[] for _ in range(n_clients)]
        for c in classes:
            idx_c = np.where(y == c)[0]
            rng.shuffle(idx_c)
            if alpha is None:
                props = np.ones(n_clients) / n_clients
            else:
                props = rng.dirichlet(np.repeat(alpha, n_clients))
            cuts = (np.cumsum(props) * len(idx_c)).astype(int)[:-1]
            for k, part in enumerate(np.split(idx_c, cuts)):
                idx_per_client[k] += part.tolist()
        best = idx_per_client
        if min(len(i) for i in idx_per_client) >= min_size:
            break
    return [np.array(i, dtype=int) for i in best]


def class_count_table(parts, y):
    y = np.asarray(y)
    rows = []
    for k, idx in enumerate(parts):
        row = {"Client": "Client %d" % (k + 1), "Patients": len(idx)}
        for c, name in enumerate(CLASS_NAMES):
            row[name] = int((y[idx] == c).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("Client")


parts_iid = partition_data(y_tr.values, N_CLIENTS, None, SEED)
parts_non = partition_data(y_tr.values, N_CLIENTS, ALPHA_NON_IID, SEED)
tab_iid = class_count_table(parts_iid, y_tr.values)
tab_non = class_count_table(parts_non, y_tr.values)
show_table(tab_iid, "Table 3a. Client data, IID partition", "table03a_clients_iid", cmap="Blues", fmt="{:.0f}")
show_table(tab_non, "Table 3b. Client data, Non-IID partition (Dirichlet alpha = %s)" % ALPHA_NON_IID,
           "table03b_clients_noniid", cmap="Oranges", fmt="{:.0f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for a, tab, ttl in zip(ax, [tab_iid, tab_non], ["IID partition", "Non-IID partition (alpha = %s)" % ALPHA_NON_IID]):
    tab[CLASS_NAMES].plot(kind="bar", stacked=True, ax=a, color=PALETTE[:N_CLASSES], edgecolor="black")
    a.set_title(ttl); a.set_ylabel("Patients"); a.tick_params(axis="x", rotation=0)
    for c in a.containers:
        a.bar_label(c, label_type="center", color="white", fontsize=8, fontweight="bold")
plt.tight_layout()
save_fig("fig04_iid_vs_noniid_clients")

# Effect of alpha on how different the clients are
alphas_show = [0.1, 0.5, 1.0, 10.0, None]
fig, ax = plt.subplots(1, len(alphas_show), figsize=(18, 4), sharey=True)
for a, al in zip(ax, alphas_show):
    pr = partition_data(y_tr.values, N_CLIENTS, al, SEED, min_size=3)
    tb = class_count_table(pr, y_tr.values)
    share = tb[CLASS_NAMES].div(tb[CLASS_NAMES].sum(axis=1), axis=0)
    share.plot(kind="bar", stacked=True, ax=a, color=PALETTE[:N_CLASSES], edgecolor="black", legend=False)
    a.set_title("IID" if al is None else "alpha = %s" % al)
    a.set_xticklabels(["C%d" % (i + 1) for i in range(N_CLIENTS)], rotation=0)
ax[0].set_ylabel("Class share inside client")
ax[-1].legend(CLASS_NAMES, loc="upper right", fontsize=8)
plt.suptitle("Effect of the Dirichlet alpha on client class shares", fontweight="bold", y=1.03)
plt.tight_layout()
save_fig("fig05_dirichlet_alpha_effect")

## STEPS 1, 2, 8: Central server, clients and local preprocessing
Every **client** keeps its own raw rows and does its own preprocessing:
1. sends only its local medians, the server averages them and broadcasts global medians (used only if a column is empty locally),
2. fills its own missing values using its own medians,
3. sends only sums (count, sum, sum of squares) so the server can build a global mean and standard deviation,
4. scales its own data with the global mean and standard deviation.

The **server** never sees raw patient rows of the clients. It only holds the validation and test sets.

In [ ]:
class Client:
    def __init__(self, cid, X_raw, y):
        self.cid = cid
        self.X_raw = X_raw.reset_index(drop=True)
        self.y = np.asarray(y)
        self.n = len(self.y)
        self.X = None

    def local_medians(self):
        return self.X_raw.median().values, self.X_raw.notna().sum().values

    def impute(self, global_median):
        med = self.X_raw.median().values
        med = np.where(np.isnan(med), global_median, med)
        self.X_imp = self.X_raw.fillna(dict(zip(self.X_raw.columns, med))).values.astype(float)

    def scaler_stats(self):
        return self.n, self.X_imp.sum(axis=0), (self.X_imp ** 2).sum(axis=0)

    def apply_scaler(self, mean, std):
        self.X = (self.X_imp - mean) / std


def federated_preprocessing(clients, X_val_raw, X_test_raw):
    cols = clients[0].X_raw.columns
    meds, cnts = zip(*[c.local_medians() for c in clients])
    meds, cnts = np.array(meds), np.array(cnts, dtype=float)
    w = np.where(np.isnan(meds), 0, cnts)
    global_median = (np.nan_to_num(meds) * w).sum(axis=0) / np.maximum(w.sum(axis=0), 1)
    for c in clients:
        c.impute(global_median)
    stats = [c.scaler_stats() for c in clients]
    N = sum(s[0] for s in stats)
    mean = sum(s[1] for s in stats) / N
    var = sum(s[2] for s in stats) / N - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-8))
    for c in clients:
        c.apply_scaler(mean, std)
    prep = lambda Xdf: (Xdf.fillna(dict(zip(cols, global_median))).values.astype(float) - mean) / std
    return prep(X_val_raw), prep(X_test_raw)


_cache = {}


def prepare_experiment(seed, alpha):
    key = (seed, alpha)
    if key in _cache:
        return _cache[key]
    Xtr, Xva, Xte, ytr, yva, yte = holdout_split(X_all, y_all, seed)
    parts = partition_data(ytr.values, N_CLIENTS, alpha, seed)
    clients = [Client(k, Xtr.iloc[idx], ytr.values[idx]) for k, idx in enumerate(parts)]
    Xva_s, Xte_s = federated_preprocessing(clients, Xva, Xte)
    data = dict(clients=clients, X_val=Xva_s, y_val=yva.values, X_test=Xte_s, y_test=yte.values)
    _cache[key] = data
    return data


demo = prepare_experiment(SEED, ALPHA_NON_IID)
print("Clients ready:", [c.n for c in demo["clients"]], "patients | features:", demo["X_val"].shape[1])
print("Missing values left after local preprocessing:", int(np.isnan(np.vstack([c.X for c in demo["clients"]])).sum()))

## STEPS 9, 10, 11: Local training, local epochs, weight updates
The model is a small neural network (one hidden layer) written in plain NumPy, so nothing is hidden.
A client trains the model on its own data for `LOCAL_EPOCHS` epochs and then computes the **weight update**
(new weights minus the weights it received).

In [ ]:
class MLP:
    def __init__(self, n_in, n_hidden, n_out, seed=0):
        rng = np.random.RandomState(seed)
        self.shapes = [(n_in, n_hidden), (n_hidden,), (n_hidden, n_out), (n_out,)]
        self.sizes = [int(np.prod(s)) for s in self.shapes]
        self.w = np.zeros(sum(self.sizes))           # all weights stored in ONE flat vector
        parts = self._unpack(self.w)
        parts[0][:] = rng.randn(n_in, n_hidden) * np.sqrt(2.0 / n_in)
        parts[2][:] = rng.randn(n_hidden, n_out) * np.sqrt(2.0 / n_hidden)

    def _unpack(self, w):
        out, i = [], 0
        for s, n in zip(self.shapes, self.sizes):
            out.append(w[i:i + n].reshape(s))
            i += n
        return out

    def predict_proba(self, X):
        W1, b1, W2, b2 = self._unpack(self.w)
        h = np.maximum(0, X @ W1 + b1)
        z = h @ W2 + b2
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def gradient(self, X, y, sample_w):
        W1, b1, W2, b2 = self._unpack(self.w)
        h = np.maximum(0, X @ W1 + b1)
        z = h @ W2 + b2
        z = z - z.max(axis=1, keepdims=True)
        p = np.exp(z); p = p / p.sum(axis=1, keepdims=True)
        Y = np.eye(b2.size)[y]
        dz = (p - Y) * sample_w[:, None] / sample_w.sum()
        dW2 = h.T @ dz; db2 = dz.sum(axis=0)
        dh = dz @ W2.T; dh[h <= 0] = 0
        dW1 = X.T @ dh; db1 = dh.sum(axis=0)
        dW1 = dW1 + L2 * W1; dW2 = dW2 + L2 * W2
        return np.concatenate([dW1.ravel(), db1.ravel(), dW2.ravel(), db2.ravel()])


def balanced_sample_weights(y, n_classes):
    counts = np.bincount(y, minlength=n_classes).astype(float)
    cw = np.where(counts > 0, len(y) / (n_classes * np.maximum(counts, 1)), 0.0)
    return cw[y]


def train_local(model, X, y, epochs, lr, batch, rng, mu=0.0, w_global=None):
    # Plain mini-batch gradient descent. mu > 0 adds the FedProx term that keeps weights near the global ones.
    sw_all = balanced_sample_weights(y, N_CLASSES)
    steps = 0
    for _ in range(epochs):
        order = rng.permutation(len(y))
        for s in range(0, len(y), batch):
            idx = order[s:s + batch]
            g = model.gradient(X[idx], y[idx], sw_all[idx] + 1e-12)
            if mu > 0 and w_global is not None:
                g = g + mu * (model.w - w_global)
            model.w -= lr * g
            steps += 1
    return steps


print("Neural network ready. Number of weights:", MLP(len(FEATURES), HIDDEN_UNITS, N_CLASSES).w.size)

## STEPS 12, 13, 14: Quantization, differential privacy, secure upload
Before a client uploads its update:
* **Differential privacy (step 13)**: the update is clipped to norm `DP_CLIP`; Gaussian noise (scaled by epsilon and delta) is added to the aggregated sum.
* **Quantization (step 12)**: the update is rounded to `QUANT_BITS` bits (8-bit numbers instead of 32-bit).
* **Secure upload (step 14)**: pairs of clients add and subtract the same random mask, so the server only sees random-looking vectors, but the masks cancel out when everything is summed (secure aggregation).

In [ ]:
def clip_update(v, C):
    norm = np.linalg.norm(v)
    return v * min(1.0, C / (norm + 1e-12))


def quantize_update(v, bits):
    levels = 2 ** (bits - 1) - 1
    mx = np.abs(v).max()
    if mx == 0:
        return v
    scale = mx / levels
    return np.round(v / scale) * scale


def add_secure_masks(updates, rng, scale=5.0):
    masked = [u.copy() for u in updates]
    for i in range(len(updates)):
        for j in range(i + 1, len(updates)):
            mask = rng.normal(0, scale, size=updates[0].shape)
            masked[i] += mask
            masked[j] -= mask
    return masked


def dp_noise_multiplier(epsilon, delta):
    return math.sqrt(2 * math.log(1.25 / delta)) / epsilon


# Small demonstration for the paper: what the server sees with and without secure upload
rng_demo = np.random.RandomState(0)
true_updates = [rng_demo.normal(0, 0.05, 674) for _ in range(5)]
masked_updates = add_secure_masks(true_updates, rng_demo)
print("Largest difference between the true sum and the masked sum: %.2e (masks cancel)"
      % np.abs(np.sum(true_updates, axis=0) - np.sum(masked_updates, axis=0)).max())

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(true_updates[0], bins=40, color=PALETTE[0], edgecolor="black")
ax[0].set_title("A client update (before masking)")
ax[1].hist(masked_updates[0], bins=40, color=PALETTE[3], edgecolor="black")
ax[1].set_title("What the server sees (masked)")
q = quantize_update(true_updates[0], QUANT_BITS)
ax[2].scatter(true_updates[0], q, s=10, color=PALETTE[2])
ax[2].plot([-0.2, 0.2], [-0.2, 0.2], "k--", lw=1)
ax[2].set_title("%d-bit quantization: original vs quantized" % QUANT_BITS)
ax[2].set_xlabel("Original value"); ax[2].set_ylabel("Quantized value")
plt.tight_layout()
save_fig("fig06_secure_upload_and_quantization")

n_w = MLP(len(FEATURES), HIDDEN_UNITS, N_CLASSES).w.size
comm = pd.DataFrame({
    "Upload per client per round (KB)": [n_w * 4 / 1024, n_w * QUANT_BITS / 8 / 1024],
}, index=["32-bit floats (no quantization)", "%d-bit quantized" % QUANT_BITS])
comm["Total upload, all clients, all rounds (MB)"] = comm.iloc[:, 0] * N_CLIENTS * ROUNDS / 1024
show_table(comm, "Table 4. Communication cost with and without quantization", "table04_communication_cost", cmap="Purples", fmt="{:.3f}")

## STEPS 15 to 19: Communication rounds, aggregation, global update, broadcast, convergence
In every **communication round**:
1. the server **broadcasts** the global weights (step 18),
2. each client trains locally and creates a protected update (steps 9 to 14),
3. the server **aggregates** the updates with **FedAvg**, **FedProx** or **FedNova** (step 16),
4. the server makes the **global model update** (step 17) and evaluates it (step 20).

The round with the lowest validation loss is kept as the final global model (the test set is never used for this choice).

In [ ]:
def evaluate_model(model, X, y):
    proba = model.predict_proba(X)
    pred = proba.argmax(axis=1)
    if N_CLASSES == 2:
        auc = roc_auc_score(y, proba[:, 1])
    else:
        auc = roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=list(range(N_CLASSES)))
    return {"Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, average="weighted", zero_division=0),
            "Recall": recall_score(y, pred, average="weighted", zero_division=0),
            "F1-score": f1_score(y, pred, average="weighted", zero_division=0),
            "AUC": auc}


def val_loss_of(model, X, y):
    return log_loss(y, model.predict_proba(X), labels=list(range(N_CLASSES)))


def run_federated(data, aggregator="FedAvg", quant=True, dp=True, epsilon=DP_EPSILON, secure=True, seed=0):
    rng = np.random.RandomState(seed)
    clients = data["clients"]
    m = len(clients)
    n_in = data["X_val"].shape[1]
    server_model = MLP(n_in, HIDDEN_UNITS, N_CLASSES, seed)         # STEP 1: the central server holds the global model
    n_total = sum(c.n for c in clients)
    p = np.array([c.n / n_total for c in clients])                  # client weights in the average
    sigma = dp_noise_multiplier(epsilon, DP_DELTA)
    tau_ref = np.mean([LOCAL_EPOCHS * math.ceil(c.n / BATCH_SIZE) for c in clients])

    history, best = [], {"loss": np.inf, "w": None, "round": 0}
    for r in range(1, ROUNDS + 1):                                  # STEP 15: communication rounds
        w_global = server_model.w.copy()                            # STEP 18: broadcast
        uploads, taus = [], []
        for k, c in enumerate(clients):
            local = MLP(n_in, HIDDEN_UNITS, N_CLASSES, seed)
            local.w = w_global.copy()
            mu = MU_PROX if aggregator == "FedProx" else 0.0
            steps = train_local(local, c.X, c.y, LOCAL_EPOCHS, LEARNING_RATE, BATCH_SIZE, rng, mu, w_global)  # STEPS 9, 10
            delta = local.w - w_global                              # STEP 11: weight update
            if aggregator == "FedNova":
                delta = delta / steps * tau_ref                     # remove the effect of different numbers of local steps
                taus.append(steps)
            if dp:
                delta = clip_update(delta, DP_CLIP)                 # STEP 13: clipping
            if quant:
                delta = quantize_update(delta, QUANT_BITS)          # STEP 12: quantization
            uploads.append(p[k] * delta)
        if secure:
            uploads = add_secure_masks(uploads, rng)                # STEP 14: secure upload
        agg = np.sum(uploads, axis=0)                               # STEP 16: aggregation (weighted sum)
        if dp:
            agg = agg + rng.normal(0, sigma * DP_CLIP * p.max(), size=agg.size)   # STEP 13: Gaussian noise
        if aggregator == "FedNova":
            agg = agg * (np.sum(p * np.array(taus)) / tau_ref)
        server_model.w = w_global + SERVER_LR * agg                 # STEP 17: global model update

        vl = val_loss_of(server_model, data["X_val"], data["y_val"])
        va = evaluate_model(server_model, data["X_val"], data["y_val"])["Accuracy"]
        ta = evaluate_model(server_model, data["X_test"], data["y_test"])["Accuracy"]
        history.append({"Round": r, "Val_Loss": vl, "Val_Accuracy": va, "Test_Accuracy": ta})
        if vl < best["loss"]:
            best = {"loss": vl, "w": server_model.w.copy(), "round": r}

    hist = pd.DataFrame(history)
    final = MLP(n_in, HIDDEN_UNITS, N_CLASSES, seed)
    final.w = best["w"].copy()
    # STEP 19: convergence = first round that reaches 99% of the best validation accuracy
    conv_round = int(hist.loc[hist["Val_Accuracy"] >= 0.99 * hist["Val_Accuracy"].max(), "Round"].iloc[0])
    return {"model": final, "history": hist, "best_round": best["round"], "conv_round": conv_round,
            "val": evaluate_model(final, data["X_val"], data["y_val"]),
            "test": evaluate_model(final, data["X_test"], data["y_test"])}


t0 = time.time()
_r = run_federated(demo, "FedAvg", seed=SEED)
print("One full federated run took %.1f seconds. Test accuracy: %.4f" % (time.time() - t0, _r["test"]["Accuracy"]))

## Reference models (centralized, no federation)
These pooled-data models are only a reference so readers can see the cost (or gain) of federated and private training.

In [ ]:
def run_centralized(data, seed):
    Xc = np.vstack([c.X for c in data["clients"]])
    yc = np.concatenate([c.y for c in data["clients"]])
    rng = np.random.RandomState(seed)
    out = {}

    mlp = MLP(Xc.shape[1], HIDDEN_UNITS, N_CLASSES, seed)
    best_loss, best_w = np.inf, None
    for ep in range(150):
        train_local(mlp, Xc, yc, 1, LEARNING_RATE, BATCH_SIZE, rng)
        vl = val_loss_of(mlp, data["X_val"], data["y_val"])
        if vl < best_loss:
            best_loss, best_w = vl, mlp.w.copy()
    mlp.w = best_w
    out["Centralized MLP"] = evaluate_model(mlp, data["X_test"], data["y_test"])

    class Wrap:
        def __init__(self, m): self.m = m
        def predict_proba(self, X): return self.m.predict_proba(X)
    for name, clf in [("Centralized Random Forest", RandomForestClassifier(300, class_weight="balanced", random_state=seed)),
                      ("Centralized Logistic Regression", LogisticRegression(max_iter=2000, class_weight="balanced"))]:
        clf.fit(Xc, yc)
        out[name] = evaluate_model(Wrap(clf), data["X_test"], data["y_test"])
    return out

## Run all experiments (this is the main part and takes a few minutes)
For every seed we run: centralized references, and FedAvg / FedProx / FedNova on **IID** and **Non-IID** data,
always with quantization, differential privacy and secure upload switched on.

In [ ]:
SEEDS = [SEED + i for i in range(N_REPEATS)]
SETTINGS = {"IID": None, "Non-IID (alpha=%s)" % ALPHA_NON_IID: ALPHA_NON_IID}
NON_NAME = "Non-IID (alpha=%s)" % ALPHA_NON_IID

store, rows = {}, []
t0 = time.time()
for sd in SEEDS:
    for sname, alpha in SETTINGS.items():
        data = prepare_experiment(sd, alpha)
        for agg in AGGREGATORS:
            res = run_federated(data, agg, seed=sd)
            store[(sd, sname, agg)] = res
            row = {"Seed": sd, "Setting": sname, "Method": agg, "Val_Accuracy": res["val"]["Accuracy"],
                   "Rounds_to_converge": res["conv_round"]}
            row.update({"Test_" + k: v for k, v in res["test"].items()})
            rows.append(row)
    cen = run_centralized(prepare_experiment(sd, None), sd)
    for name, mt in cen.items():
        row = {"Seed": sd, "Setting": "Centralized (no FL)", "Method": name, "Val_Accuracy": np.nan, "Rounds_to_converge": np.nan}
        row.update({"Test_" + k: v for k, v in mt.items()})
        rows.append(row)
    print("Seed %d finished (%.0f s)" % (sd, time.time() - t0))

results = pd.DataFrame(rows)
print("All experiments finished.")

## STEP 16 and 19: Aggregation methods compared, and convergence curves

In [ ]:
METRICS = ["Test_Accuracy", "Test_Precision", "Test_Recall", "Test_F1-score", "Test_AUC"]


def mean_std_table(df):
    g = df.groupby(["Setting", "Method"], sort=False)
    mean, std = g[METRICS].mean(), g[METRICS].std()
    text = mean.copy().astype(object)
    for c in METRICS:
        text[c] = [("%.4f +- %.4f" % (a, b)) for a, b in zip(mean[c], std[c])]
    mean.columns = [c.replace("Test_", "") for c in mean.columns]
    text.columns = mean.columns
    return mean, text


perf_mean, perf_text = mean_std_table(results)
show_table(perf_mean, "Table 5. Global test performance, mean over %d runs" % N_REPEATS, "table05_performance_mean", cmap="YlGnBu")
print("\nTable 5b. Same table as mean +- standard deviation (copy this into the paper)")
perf_text.to_csv(os.path.join(OUT_DIR, "tables", "table05b_performance_mean_std.csv"))
display(perf_text)

conv = (results[results["Setting"] != "Centralized (no FL)"]
        .groupby(["Setting", "Method"], sort=False)[["Val_Accuracy", "Rounds_to_converge"]].mean())
conv.columns = ["Mean validation accuracy", "Mean rounds to converge"]
show_table(conv, "Table 6. Convergence speed of the aggregation methods", "table06_convergence", cmap="BuGn", fmt="{:.3f}")

# Bar chart: aggregation methods
fed = results[results["Setting"] != "Centralized (no FL)"]
plt.figure(figsize=(9, 5))
sns.barplot(data=fed, x="Setting", y="Test_Accuracy", hue="Method", palette=PALETTE[:3], errorbar="sd",
            capsize=0.1, edgecolor="black")
plt.ylim(0.4, 1.0); plt.ylabel("Global test accuracy"); plt.title("FedAvg vs FedProx vs FedNova (mean +- std)")
plt.legend(title="Aggregation", loc="lower right")
save_fig("fig07_aggregation_comparison")

# Convergence curves (main seed)
fig, ax = plt.subplots(2, 3, figsize=(17, 8), sharex=True)
for i, sname in enumerate(SETTINGS):
    for agg, col in zip(AGGREGATORS, PALETTE[:3]):
        h = store[(SEED, sname, agg)]["history"]
        sm = lambda s: s.rolling(3, min_periods=1).mean()
        ax[i, 0].plot(h["Round"], sm(h["Val_Accuracy"]), color=col, lw=2, label=agg)
        ax[i, 1].plot(h["Round"], sm(h["Val_Loss"]), color=col, lw=2, label=agg)
        ax[i, 2].plot(h["Round"], sm(h["Test_Accuracy"]), color=col, lw=2, label=agg)
    ax[i, 0].set_ylabel(sname + "\nAccuracy")
    ax[i, 0].set_title("Validation accuracy"); ax[i, 1].set_title("Validation loss"); ax[i, 2].set_title("Global test accuracy")
ax[1, 0].set_xlabel("Communication round"); ax[1, 1].set_xlabel("Communication round"); ax[1, 2].set_xlabel("Communication round")
ax[0, 0].legend()
plt.suptitle("Convergence of the global model (3-round moving average)", fontweight="bold", y=1.0)
plt.tight_layout()
save_fig("fig08_convergence_curves")

# Pick the aggregation method with the best mean VALIDATION accuracy on Non-IID data (test set is not used for this)
val_rank = (results[results["Setting"] == NON_NAME].groupby("Method")["Val_Accuracy"].mean().sort_values(ascending=False))
BEST_AGG = val_rank.index[0]
print("Best aggregation chosen on validation data:", BEST_AGG)

## Effect of each protection layer (ablation) and the privacy-utility curve (STEPS 12, 13, 14)

In [ ]:
ablation_cfg = {
    "Plain FL (no protection)": dict(quant=False, dp=False, secure=False),
    "+ Quantization": dict(quant=True, dp=False, secure=False),
    "+ Secure upload": dict(quant=False, dp=False, secure=True),
    "+ Differential privacy": dict(quant=False, dp=True, secure=False),
    "All protections": dict(quant=True, dp=True, secure=True),
}
abl_rows = []
for sd in SEEDS:
    data = prepare_experiment(sd, ALPHA_NON_IID)
    for name, cfg in ablation_cfg.items():
        res = run_federated(data, BEST_AGG, seed=sd, **cfg)
        abl_rows.append({"Variant": name, "Seed": sd, "Accuracy": res["test"]["Accuracy"], "F1-score": res["test"]["F1-score"], "AUC": res["test"]["AUC"]})
abl = pd.DataFrame(abl_rows)
abl_mean = abl.groupby("Variant", sort=False)[["Accuracy", "F1-score", "AUC"]].mean()
show_table(abl_mean, "Table 7. Ablation of quantization, secure upload and differential privacy (%s, Non-IID)" % BEST_AGG,
           "table07_ablation", cmap="RdPu")

plt.figure(figsize=(10, 4.8))
sns.barplot(data=abl, x="Variant", y="Accuracy", palette=PALETTE, errorbar="sd", capsize=0.1, edgecolor="black")
plt.xticks(rotation=18, ha="right"); plt.ylim(0.4, 1.0); plt.title("Accuracy with different protection layers")
save_fig("fig09_ablation")

# Privacy-utility curve
eps_rows = []
for sd in SEEDS:
    data = prepare_experiment(sd, ALPHA_NON_IID)
    for eps in EPSILON_LIST:
        res = run_federated(data, BEST_AGG, dp=True, epsilon=eps, seed=sd)
        eps_rows.append({"Epsilon": eps, "Seed": sd, "Accuracy": res["test"]["Accuracy"], "F1-score": res["test"]["F1-score"], "AUC": res["test"]["AUC"]})
    res = run_federated(data, BEST_AGG, dp=False, seed=sd)
    eps_rows.append({"Epsilon": np.inf, "Seed": sd, "Accuracy": res["test"]["Accuracy"], "F1-score": res["test"]["F1-score"], "AUC": res["test"]["AUC"]})
eps_df = pd.DataFrame(eps_rows)
eps_mean = eps_df.groupby("Epsilon")[["Accuracy", "F1-score", "AUC"]].mean()
eps_std = eps_df.groupby("Epsilon")[["Accuracy", "F1-score", "AUC"]].std()
eps_show = eps_mean.copy(); eps_show.index = ["no DP" if np.isinf(e) else "%g" % e for e in eps_show.index]
show_table(eps_show, "Table 8. Privacy-utility trade-off (smaller epsilon = stronger privacy)", "table08_privacy_utility", cmap="OrRd")

plt.figure(figsize=(9, 5))
xs = [(max(EPSILON_LIST) * 2 if np.isinf(e) else e) for e in eps_mean.index]
for metric, col in zip(["Accuracy", "F1-score", "AUC"], PALETTE[:3]):
    plt.errorbar(xs, eps_mean[metric], yerr=eps_std[metric], marker="o", lw=2, capsize=4, color=col, label=metric)
plt.xscale("log")
plt.xticks(xs, ["%g" % e if not np.isinf(e) else "no DP" for e in eps_mean.index])
plt.xlabel("Privacy budget epsilon (per round)"); plt.ylabel("Test score")
plt.title("Privacy-utility trade-off"); plt.legend()
save_fig("fig10_privacy_utility_curve")

## STEPS 20 and 21: Global evaluation and performance metrics
The final global model (best round, chosen on validation data) is evaluated once on the untouched global test set.

In [ ]:
final = store[(SEED, NON_NAME, BEST_AGG)]
final_model, data_main = final["model"], prepare_experiment(SEED, ALPHA_NON_IID)
X_test_s, y_test_np = data_main["X_test"], data_main["y_test"]
proba_test = final_model.predict_proba(X_test_s)
pred_test = proba_test.argmax(axis=1)

print("Final model: %s on Non-IID data | best round: %d | global test accuracy: %.4f" %
      (BEST_AGG, final["best_round"], final["test"]["Accuracy"]))

# Table of final metrics (validation and test)
final_tab = pd.DataFrame({"Validation": final["val"], "Global test": final["test"]}).T
show_table(final_tab, "Table 9. Final global model performance (seed %d)" % SEED, "table09_final_performance", cmap="YlOrRd")

# Per-class report
rep = pd.DataFrame(classification_report(y_test_np, pred_test, target_names=CLASS_NAMES, output_dict=True, zero_division=0)).T
show_table(rep, "Table 10. Per-class classification report (global test set)", "table10_classification_report", cmap="Greens")

# Confusion matrix (counts and percentages)
cm = confusion_matrix(y_test_np, pred_test, labels=list(range(N_CLASSES)))
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[0], cbar=False,
            annot_kws={"size": 16, "weight": "bold"}, linewidths=1, linecolor="white")
ax[0].set_title("Confusion matrix (counts)"); ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")
sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Oranges", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1], cbar=False,
            annot_kws={"size": 16, "weight": "bold"}, linewidths=1, linecolor="white")
ax[1].set_title("Confusion matrix (% of each actual class)"); ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Actual")
plt.tight_layout()
save_fig("fig11_confusion_matrix")

# ROC curve
plt.figure(figsize=(7, 6))
if N_CLASSES == 2:
    fpr, tpr, _ = roc_curve(y_test_np, proba_test[:, 1])
    plt.plot(fpr, tpr, color=PALETTE[3], lw=3, label="Global model (AUC = %.3f)" % final["test"]["AUC"])
else:
    for c, col in zip(range(N_CLASSES), PALETTE):
        fpr, tpr, _ = roc_curve((y_test_np == c).astype(int), proba_test[:, c])
        plt.plot(fpr, tpr, color=col, lw=2.5, label="%s (AUC = %.3f)" % (CLASS_NAMES[c], roc_auc_score((y_test_np == c).astype(int), proba_test[:, c])))
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curve on the global test set"); plt.legend(loc="lower right")
save_fig("fig12_roc_curve")

# Metric bar chart: federated vs centralized
cmp = perf_mean[["Accuracy", "Precision", "Recall", "F1-score", "AUC"]]
cmp.plot(kind="bar", figsize=(13, 5.5), color=PALETTE[:5], edgecolor="black", width=0.8)
plt.ylim(0.3, 1.0); plt.ylabel("Score"); plt.title("All methods: federated (with protections) vs centralized reference")
plt.xticks(rotation=35, ha="right"); plt.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.45))
save_fig("fig13_all_methods_comparison")

## STEP 22: Explainable AI (XAI)
The explanations are made for the **final global federated model** and explain the predicted **risk of death**.
* SHAP background data comes from the validation set, and explanations are computed for the test patients.
* Five figures and four tables are produced: SHAP summary, SHAP importance bars, SHAP single-patient waterfall, LIME, and permutation importance.

In [ ]:
def risk_fn(X):
    return final_model.predict_proba(X)[:, POS_CLASS]


background = shap.kmeans(data_main["X_val"], 15)
explainer = shap.KernelExplainer(risk_fn, background)
shap_values = np.array(explainer.shap_values(X_test_s, nsamples=300, silent=True))
if shap_values.ndim == 3:
    shap_values = shap_values[..., 0]
base_value = float(np.ravel(explainer.expected_value)[0])
print("SHAP values computed:", shap_values.shape)

# ---- XAI Figure 1: SHAP summary (beeswarm) ----
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, X_test_s, feature_names=FEATURES, show=False, max_display=14)
plt.title("SHAP summary plot: effect of each feature on death risk")
save_fig("fig14_xai_shap_summary")

# ---- XAI Table 1 + Figure 2: global SHAP importance ----
shap_imp = pd.DataFrame({"Feature": FEATURES, "Mean |SHAP|": np.abs(shap_values).mean(axis=0)}).sort_values("Mean |SHAP|", ascending=False)
shap_imp["Rank"] = range(1, len(shap_imp) + 1)
shap_imp = shap_imp.set_index("Rank")
show_table(shap_imp.head(10), "Table 11. Top 10 features by mean absolute SHAP value", "table11_xai_shap_importance", cmap="YlOrBr", fmt="{:.4f}")

top = shap_imp.head(12).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top["Feature"], top["Mean |SHAP|"], color=sns.color_palette("viridis", len(top)), edgecolor="black")
for i, v in enumerate(top["Mean |SHAP|"]):
    plt.text(v + 0.0005, i, "%.3f" % v, va="center", fontsize=9)
plt.xlabel("Mean |SHAP value|"); plt.title("Global feature importance (SHAP)")
save_fig("fig15_xai_shap_bar")

# ---- XAI Figure 3: SHAP waterfall for one high-risk patient ----
case = int(np.argmax(risk_fn(X_test_s)))
exp_obj = shap.Explanation(values=shap_values[case], base_values=base_value, data=X_test_s[case], feature_names=FEATURES)
plt.figure()
shap.plots.waterfall(exp_obj, max_display=12, show=False)
plt.title("SHAP waterfall for one test patient (predicted risk %.2f, actual: %s)" %
          (risk_fn(X_test_s[case:case + 1])[0], CLASS_NAMES[y_test_np[case]]), fontsize=11)
save_fig("fig16_xai_shap_waterfall")

case_tab = pd.DataFrame({"Feature": FEATURES, "Standardized value": X_test_s[case], "SHAP value": shap_values[case]})
case_tab["Pushes risk"] = np.where(case_tab["SHAP value"] > 0, "Up", "Down")
case_tab = case_tab.reindex(case_tab["SHAP value"].abs().sort_values(ascending=False).index).head(8).reset_index(drop=True)
show_table(case_tab.set_index("Feature"), "Table 12. Local explanation of the patient in the waterfall plot", "table12_xai_local_explanation", cmap="coolwarm", fmt="{:.3f}")

# ---- XAI Figure 4: LIME ----
lime_explainer = LimeTabularExplainer(data_main["X_val"], feature_names=FEATURES, class_names=CLASS_NAMES,
                                      mode="classification", discretize_continuous=True, random_state=SEED)
lime_exp = lime_explainer.explain_instance(X_test_s[case], final_model.predict_proba, num_features=10, labels=(POS_CLASS,), num_samples=2000)
fig = lime_exp.as_pyplot_figure(label=POS_CLASS)
fig.set_size_inches(9, 5.5)
plt.title("LIME explanation of the same patient (class: %s)" % CLASS_NAMES[POS_CLASS])
save_fig("fig17_xai_lime")

# LIME global view: average absolute weight over 25 test patients
lime_sum = np.zeros(len(FEATURES))
n_lime = min(25, len(X_test_s))
for i in range(n_lime):
    e = lime_explainer.explain_instance(X_test_s[i], final_model.predict_proba, num_features=len(FEATURES), labels=(POS_CLASS,), num_samples=1000)
    for idx, wt in e.as_map()[POS_CLASS]:
        lime_sum[idx] += abs(wt)
lime_imp = lime_sum / n_lime

# ---- XAI Figure 5 + Table: permutation importance ----
def permutation_importance(model_fn, X, y, repeats=30, seed=0):
    rng = np.random.RandomState(seed)
    base = accuracy_score(y, model_fn(X).argmax(axis=1))
    mean_drop, std_drop = [], []
    for j in range(X.shape[1]):
        drops = []
        for _ in range(repeats):
            Xp = X.copy()
            Xp[:, j] = rng.permutation(Xp[:, j])
            drops.append(base - accuracy_score(y, model_fn(Xp).argmax(axis=1)))
        mean_drop.append(np.mean(drops)); std_drop.append(np.std(drops))
    return np.array(mean_drop), np.array(std_drop)


perm_mean, perm_std = permutation_importance(final_model.predict_proba, X_test_s, y_test_np)
perm = pd.DataFrame({"Feature": FEATURES, "Accuracy drop": perm_mean, "Std": perm_std}).sort_values("Accuracy drop", ascending=False)
top = perm.head(12).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top["Feature"], top["Accuracy drop"], xerr=top["Std"], color=sns.color_palette("magma", len(top)), edgecolor="black", capsize=3)
plt.axvline(0, color="black", lw=1)
plt.xlabel("Drop in test accuracy when the feature is shuffled"); plt.title("Permutation feature importance")
save_fig("fig18_xai_permutation_importance")

# ---- XAI Table: all three methods side by side ----
cmp_xai = pd.DataFrame({"Feature": FEATURES,
                        "SHAP (mean |value|)": np.abs(shap_values).mean(axis=0),
                        "LIME (mean |weight|)": lime_imp,
                        "Permutation (accuracy drop)": perm_mean}).set_index("Feature")
for c in list(cmp_xai.columns):
    cmp_xai[c.split(" (")[0] + " rank"] = cmp_xai[c].rank(ascending=False).astype(int)
cmp_xai["Average rank"] = cmp_xai[[c for c in cmp_xai.columns if c.endswith("rank")]].mean(axis=1)
cmp_xai = cmp_xai.sort_values("Average rank")
show_table(cmp_xai.head(10), "Table 13. Top 10 features: agreement of SHAP, LIME and permutation importance", "table13_xai_method_comparison", cmap="PuBuGn", fmt="{:.3f}")

## Final summary and download of all results

In [ ]:
print("=" * 70)
print("FINAL RESULT (global federated model, %s, Non-IID alpha=%s, %d clients)" % (BEST_AGG, ALPHA_NON_IID, N_CLIENTS))
print("=" * 70)
for k, v in final["test"].items():
    print("%-10s : %.4f" % (k, v))
print("\nMean over %d repeated runs:" % N_REPEATS)
print(perf_text.loc[NON_NAME].loc[BEST_AGG].to_string())
print("\nTop 5 features (average rank of SHAP, LIME, permutation):", cmp_xai.head(5).index.tolist())

results.to_csv(os.path.join(OUT_DIR, "tables", "all_runs_raw_results.csv"), index=False)
zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("\nAll figures (PNG, 300 dpi) and tables (CSV) saved in:", OUT_DIR)
print("Zip file:", zip_path)
try:
    files.download(zip_path)
except Exception as e:
    print("Use the Files panel on the left to download the zip file.")